# 01 · NumPy foundations

Pairs with `docs/08-data-layer-lab.md` (Parts A-B). Goal: understand every NumPy
idea the preprocessing code uses, by writing it yourself.

In [ ]:
import numpy as np

## 1. An array is data + shape + dtype + strides

`shape` is how you index it; `dtype` is how many bytes per element; `strides` is how
many **bytes** to step per axis. Reshapes/slices/transposes usually make a new
*view* over the same memory.

In [ ]:
x = np.arange(12).reshape(3, 4).astype(np.float64)
print("shape", x.shape, "| dtype", x.dtype, "| itemsize", x.itemsize, "bytes")
print("strides", x.strides, "| total bytes", x.nbytes)

In [ ]:
# Your turn: make x.T (transpose) and a slice x[:, ::2].
# Check whether each SHARES memory with x using np.shares_memory(a, b).
# TODO

> **Concepts to note** (copy into your own theory notebook):
> - View vs copy: transpose/slice/reshape are usually views (no data copied).
> - `np.shares_memory(a, b)` tells you if two arrays alias the same buffer.
> - This is the *same* idea as PyTorch tensor storage/strides (GUIDE Step 1).

## 2. Axes and reductions

`axis=0` moves down rows; `axis=1` across columns. For our curves shaped
`(n_wells, n_frames)`, `axis=1` means "per well, along time".

In [ ]:
a = np.array([[1., 2., 3.], [10., 20., 30.]])
print("sum all      ", a.sum())
print("sum axis=0   ", a.sum(axis=0))   # per column
print("max axis=1   ", a.max(axis=1))   # per row  <- what mining uses

In [ ]:
# Your turn: given a (5, 8) array, get the argmax column index of each row,
# restricted to columns 2..6 only.  Hint: 2 + a[:, 2:6].argmax(axis=1)
# TODO

## 3. Broadcasting and `np.newaxis`

Broadcasting aligns shapes from the **right**. A length-1 axis stretches to match.
This is exactly how AUC-normalization divides each curve by its own scalar area.

In [ ]:
roi = np.random.default_rng(0).random((4, 6))     # 4 curves x 6 points
area = roi.sum(axis=1)                              # shape (4,)
print("roi", roi.shape, "area", area.shape)
# roi / area FAILS to align; add an axis so (4,6)/(4,1) broadcasts across columns:
norm = roi / area[:, np.newaxis]
print("row sums after norm:", norm.sum(axis=1))    # all ~1

In [ ]:
# Your turn: subtract each row's MEAN from every element using broadcasting.
# TODO

> **Concepts to note** (copy into your own theory notebook):
> - Broadcasting rule: compare shapes right-to-left; dims must be equal or 1.
> - `np.newaxis` (== `None`) inserts a length-1 axis; `a[:, None]` is the idiom.
> - Identical rule in PyTorch — learn it once here.

## 4. Boolean masks and `where`

Comparisons make boolean arrays; combine with `&` / `|` (never `and`/`or`).
`np.flatnonzero(mask)` gives the integer indices of the `True` entries — this is how
`positive_mask` selects wells.

In [ ]:
v = np.array([1., 5., 2., 9., 0.])
mask = (v > 1) & (v < 9)
print("mask", mask)
print("kept indices", np.flatnonzero(mask))   # == np.where(mask)[0]
print("kept values ", v[mask])

In [ ]:
# Your turn: from a (6, 10) random array, keep rows whose max > 0.8.
# Return the row indices.  # TODO

## 5. Finite differences and integration

`np.gradient` = derivative by central differences (edges one-sided), same length as
input. `np.trapezoid` = area under a curve (trapezoidal rule). The melt pipeline uses
`-np.gradient(...)` so a falling fluorescence curve gives a positive melt peak.

In [ ]:
t = np.linspace(0, 10, 200)
y = 1 / (1 + np.exp(t - 5))            # falling sigmoid (a toy melt curve)
d = -np.gradient(y)                    # negative derivative -> positive peak
print("peak at index", d.argmax(), "| area under |d|:", np.trapezoid(d))

In [ ]:
# Your turn: verify np.trapezoid(y) matches your hand-rolled trapezoid:
#   np.sum((y[:-1] + y[1:]) / 2)   (unit spacing)
# TODO

> **Concepts to note** (copy into your own theory notebook):
> - `np.gradient` central differences; `np.trapezoid` trapezoidal integration.
> - Why negate the gradient for melt curves? (falling signal -> positive peak)
> - Self-check: what does `axis=` do to the output shape of a reduction?